# Section for Full TMDB Movies Dataset 2024 
(BRONZE >> SILVER)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Paths
BRONZE_TABLE_PATH = "mvpde.1_bronze.film"
SILVER_TABLE_PATH = "mvpde.2_silver.film"

# -------------------------------------------------------------------------
# Read Raw Bronze Data
# -------------------------------------------------------------------------
film_bronze = spark.read.format("delta").table(BRONZE_TABLE_PATH)

# Define reasonable lower and upper boundaries
MIN_RELEASE_DATE = "1888-01-01"
MAX_RELEASE_DATE = F.date_add(F.current_date(), 365 * 2)  # Current date + 2 years for announced films

# Helper: Clean text fields (trim whitespace and map empty/string nulls to SQL NULL)
def clean_str(col_name):
    return F.when(
        (F.trim(F.col(col_name)) == "") | 
        (F.lower(F.trim(F.col(col_name))).isin("null", "none", "nan")), 
        None
    ).otherwise(F.trim(F.col(col_name)))

# Helper: Convert zero values in numerical columns to NULL
def zero_to_null(col_ref):
    return F.when(col_ref == 0, None).otherwise(col_ref)

# Helper: Convert delimited strings into ArrayType(StringType)
def parse_array(col_name, delimiter=r"\s*,\s*"):
    cleaned = clean_str(col_name)
    return F.when(cleaned.isNotNull(), F.split(cleaned, delimiter)).otherwise(None)

# -------------------------------------------------------------------------
# Apply Transformations: Casting, Cleaning & Native Array Parsing
# -------------------------------------------------------------------------
film_bronze_clean_cast = (film_bronze
    # Primary Key Validation & Casting
    .withColumn("id", clean_str("id").cast("long"))
    .filter(F.col("id").isNotNull())
    
    # Text Fields
    .withColumn("title", clean_str("title"))
    .withColumn("original_title", clean_str("original_title"))
    .withColumn("overview", clean_str("overview"))
    .withColumn("tagline", clean_str("tagline"))
    .withColumn("homepage", clean_str("homepage"))
    .withColumn("imdb_id", clean_str("imdb_id"))
    .withColumn("backdrop_path", clean_str("backdrop_path"))
    .withColumn("poster_path", clean_str("poster_path"))
    .withColumn("status", clean_str("status"))
    .withColumn("original_language", clean_str("original_language"))
    
    # Numeric Measures
    .withColumn("vote_average", clean_str("vote_average").cast("double"))
    .withColumn("vote_count", clean_str("vote_count").cast("long"))
    .withColumn("popularity", clean_str("popularity").cast("double"))
    .withColumn("runtime", zero_to_null(clean_str("runtime").cast("int")))
    .withColumn("budget", zero_to_null(clean_str("budget").cast("long")))
    .withColumn("revenue", zero_to_null(clean_str("revenue").cast("long")))
    
    # Boolean Flag
    .withColumn(
        "adult", 
        F.when(F.lower(clean_str("adult")).isin("true", "1", "t"), True)
         .when(F.lower(clean_str("adult")).isin("false", "0", "f"), False)
         .otherwise(None)
    )
    
    # Date Field
    .withColumn("release_date", F.to_date(clean_str("release_date"), "yyyy-MM-dd"))
    # 2. Filter out nulls, historical impossible dates, and far future dates
    .filter(
        F.col("release_date").isNotNull() &
        (F.col("release_date") >= F.lit(MIN_RELEASE_DATE)) &
        (F.col("release_date") <= MAX_RELEASE_DATE)
    )
    
    # Native Array Fields
    .withColumn("genres", parse_array("genres"))
    .withColumn("production_companies", parse_array("production_companies"))
    .withColumn("production_countries", parse_array("production_countries"))
    .withColumn("spoken_languages", parse_array("spoken_languages"))
    .withColumn("keywords", parse_array("keywords"))
    
    # Audit Metadata Strategy
    .withColumnRenamed("_ingested_at", "_bronze_ingested_at")
    .withColumn("_silver_processed_at", F.current_timestamp())
    .withColumn("_source_file_name", F.col("_source_file_name"))
    .drop("_source_file_path", "_source_file_modification_time")
)

# -------------------------------------------------------------------------
# Deduplicate (Keeping 1 record per movie `id`)
# -------------------------------------------------------------------------
window_spec = Window.partitionBy("id").orderBy(
    F.col("release_date").desc_nulls_last(),
    F.col("vote_count").desc_nulls_last()
)

film_silver = film_bronze_clean_cast \
    .withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")

# -------------------------------------------------------------------------
# Write Single-Table Output to Silver Delta Lake
# -------------------------------------------------------------------------
film_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE_PATH)

print("Silver Delta load completed successfully.")

Silver Delta load completed successfully.


In [0]:
%sql
-- =========================================================================
-- 1. Table-Level Documentation
-- =========================================================================
ALTER TABLE mvpde.`2_silver`.film 
SET TAGS ('layer' = 'silver', 'domain' = 'films', 'source' = 'tmdb');

COMMENT ON TABLE mvpde.`2_silver`.film IS 
'Silver layer TMDB movies table containing cleaned, typed, and deduplicated movie metadata (1 row per movie entity). Transformed from mvpde.`1_bronze`.film .';

-- =========================================================================
-- 2. Column-Level Documentation (Identifiers & Primary Text)
-- =========================================================================
ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN id 
COMMENT 'Unique TMDB movie identifier (Primary Key, cast to BIGINT).';

ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN title 
COMMENT 'Primary localized display title of the movie.';

ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN original_title 
COMMENT 'Movie title in its original language prior to translation.';

ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN tagline 
COMMENT 'Official promotional tagline or slogan.';

ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN overview 
COMMENT 'Detailed plot summary and synopsis.';

ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN homepage 
COMMENT 'Official movie web URL.';

ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN imdb_id 
COMMENT 'Corresponding IMDb identifier (e.g., tt0137523).';

ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN status 
COMMENT 'Production release status (e.g., Released, In Production, Post Production).';

ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN original_language 
COMMENT 'ISO 639-1 primary spoken language code (e.g., en, fr, es).';

-- =========================================================================
-- 3. Column-Level Documentation (Media Paths)
-- =========================================================================
ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN backdrop_path 
COMMENT 'Relative TMDB image path for the wide horizontal backdrop poster.';

ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN poster_path 
COMMENT 'Relative TMDB image path for the primary vertical poster.';

-- =========================================================================
-- 4. Column-Level Documentation (Numerical Metrics & Dates)
-- =========================================================================
ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN vote_average 
COMMENT 'Average user rating score on a 0 to 10 scale.';

ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN vote_count 
COMMENT 'Total number of user ratings submitted on TMDB.';

ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN popularity 
COMMENT 'TMDB algorithmically computed popularity metric.';

ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN runtime 
COMMENT 'Film runtime in minutes. Zero/invalid entries mapped to NULL.';

ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN budget 
COMMENT 'Declared production budget in USD. Zero/invalid entries mapped to NULL.';

ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN revenue 
COMMENT 'Total worldwide box office revenue in USD. Zero/invalid entries mapped to NULL.';

ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN adult 
COMMENT 'Boolean flag indicating adult/18+ content.';

ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN release_date 
COMMENT 'Official theatrical or streaming release date (YYYY-MM-DD).';

-- =========================================================================
-- 5. Column-Level Documentation (Native Array Fields)
-- =========================================================================
ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN genres 
COMMENT 'ARRAY<STRING>: List of film genres (e.g., ["Action", "Drama"]). Parsed from raw comma-delimited strings.';

ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN production_companies 
COMMENT 'ARRAY<STRING>: List of production studio/company names.';

ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN production_countries 
COMMENT 'ARRAY<STRING>: List of country names where production occurred.';

ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN spoken_languages 
COMMENT 'ARRAY<STRING>: List of languages spoken in the film.';

ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN keywords 
COMMENT 'ARRAY<STRING>: Descriptive tag keywords associated with the movie.';

-- =========================================================================
-- 6. Column-Level Documentation (Silver Pipeline Audit Metadata)
-- =========================================================================
ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN _bronze_ingested_at 
COMMENT 'Timestamp when the raw payload was originally ingested into the Bronze layer.';

ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN _silver_processed_at 
COMMENT 'Timestamp when this record was processed, cleaned, and loaded into Silver.';

ALTER TABLE mvpde.`2_silver`.film ALTER COLUMN _source_file_name 
COMMENT 'Name of the original source landing file used for batch traceability.';

# Section for TMDB 5000 Movie Dataset (credits.csv)
BRONZE >> SILVER

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    ArrayType, 
    StructType, 
    StructField, 
    StringType, 
    IntegerType
)

from pyspark.sql.functions import (
    col,
    trim,
    from_json,
    current_timestamp,
    input_file_name,
    row_number
)
from pyspark.sql.window import Window

# ==========================================
# DEFINE SCHEMAS FOR JSON COLUMNS
# ==========================================
# Schema for 'cast' stringified JSON array
cast_schema = ArrayType(
    StructType([
        StructField("cast_id", IntegerType(), True),
        StructField("character", StringType(), True),
        StructField("credit_id", StringType(), True),
        StructField("gender", IntegerType(), True),
        StructField("id", IntegerType(), True),
        StructField("name", StringType(), True),
        StructField("order", IntegerType(), True)
    ])
)

# Schema for 'crew' stringified JSON array
crew_schema = ArrayType(
    StructType([
        StructField("credit_id", StringType(), True),
        StructField("department", StringType(), True),
        StructField("gender", IntegerType(), True),
        StructField("id", IntegerType(), True),
        StructField("job", StringType(), True),
        StructField("name", StringType(), True)
    ])
)

# ==========================================
# READ FROM BRONZE LAYER
# ==========================================
# Backticks are mandatory for '1_bronze' due to starting numbers and special characters
bronze_table = "`mvpde`.`1_bronze`.`credit`"
silver_table = "`mvpde`.`2_silver`.`credit`"

credit_bronze = spark.read.table(bronze_table)

# ==========================================
# CLEAN, CAST, & TRANSFORM
# ==========================================

# A. Data Quality Filter: Remove records missing primary keys
credit_filtered = credit_bronze.filter(
    col("movie_id").isNotNull() & (trim(col("movie_id")) != "")
)

# B. Deduplication: Keep the most recent record per movie_id if ingestion timestamps exist
# (Falls back to simple dropDuplicates if no ingestion sequence exists)
window_spec = Window.partitionBy("movie_id").orderBy(col("movie_id"))
credit_deduped = (
    credit_filtered
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

# C. Parsing & Type Casting
credit_silver = credit_deduped.select(
    # Cast primary key to native Integer
    col("movie_id").cast(IntegerType()).alias("movie_id"),
    
    # Standardize string formatting
    trim(col("title")).alias("title"),
    
    # Parse JSON string arrays into strongly-typed Array<Struct>
    from_json(col("cast"), cast_schema).alias("cast"),
    from_json(col("crew"), crew_schema).alias("crew"),
    
    # Audit Column: Log when the Silver pipeline processed the record
    current_timestamp().alias("processed_at")
)

# ==========================================
# WRITE TO SILVER LAYER
# ==========================================
# Delta Lake Write: Merge/Overwrite mode into the 2_silver schema
(
    credit_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

print(f"Successfully cleaned and written dataset to {silver_table}")

Successfully cleaned and written dataset to `mvpde`.`2_silver`.`credit`


In [0]:
%sql
-- =========================================================================
-- Table-Level Metadata & Tags
-- =========================================================================
ALTER TABLE mvpde.`2_silver`.credit 
SET TAGS ('layer' = 'silver', 'domain' = 'movies', 'source' = 'tmdb_5000_credits');

COMMENT ON TABLE mvpde.`2_silver`.credit IS 
'Silver layer TMDB credits table containing parsed cast and crew arrays, extracted person IDs, and audit metadata (1 record per movie_id). Transformed from Bronze layer.';

-- =========================================================================
-- Core Primary Keys & Text Fields
-- =========================================================================
ALTER TABLE mvpde.`2_silver`.credit ALTER COLUMN movie_id 
COMMENT 'Unique movie identifier linking to silver movie tables (INT, Primary Key).';

ALTER TABLE mvpde.`2_silver`.credit ALTER COLUMN title 
COMMENT 'Primary localized title of the movie (STRING).';

-- =========================================================================
-- Parsed Struct Arrays & Extracted Person ID Arrays
-- =========================================================================
ALTER TABLE mvpde.`2_silver`.credit ALTER COLUMN cast 
COMMENT 'ARRAY<STRUCT<cast_id: INT, character: STRING, credit_id: STRING, gender: INT, id: INT, name: STRING, order: INT>>: Parsed structured list of actors, roles, and ordering.';

ALTER TABLE mvpde.`2_silver`.credit ALTER COLUMN crew 
COMMENT 'ARRAY<STRUCT<credit_id: STRING, department: STRING, gender: INT, id: INT, job: STRING, name: STRING>>: Parsed structured list of crew members, departments, and jobs.';

-- =========================================================================
-- Audit Metadata Column
-- =========================================================================
ALTER TABLE mvpde.`2_silver`.credit ALTER COLUMN processed_at 
COMMENT 'TIMESTAMP: Processing timestamp when this record was cleaned, parsed, and loaded into Silver.';